# Model using Transfer Learning to learn on CIFAR Dataset 

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize pixel values
x_train, x_test = x_train / 255.0, x_test / 255.0

d:\Documents\Github\ml\courses\iitm\mtech\ai\Term-2\DA6401W\project\src\.venv\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


In [3]:
batch_size = 32

def preprocess(image, label):
    # Resize dynamically to 224x224
    image = tf.image.resize(image, (224, 224))
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)) \
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE) \
    .shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)) \
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE) \
    .batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [4]:
# Load Pretrained Model (Feature Extractor)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 21s 0us/step


In [5]:
# Add Custom Classification Head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')  # CIFAR-10 has 10 classes
])

In [6]:
# Compile Model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
history = model.fit(train_ds,
                    epochs=5,
                    validation_data=(test_ds))

Epoch 1/5


 344/1563 ━━━━━━━━━━━━━━━━━━━━ 23:41:18 70s/step - accuracy: 0.3375 - loss: 1.9142

In [ ]:
# Fine‑Tuning
# Unfreeze deeper layers for better accuracy
base_model.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),  # smaller LR
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history_ft = model.fit(train_ds, y_train,
                       epochs=3,
                       validation_data=(test_ds, y_test))